# 附：Codes .{unnumbered}

本 Notebook 用于生成 `04_tobit_model_lec_v2.ipynb` 中调用的模拟数据和图片。运行后会生成：

- `./data/tobit_credit_sim.csv`
- `./data/tobit_credit_estimates.csv`
- `./figs/limit_dep_tobit_fig01_latent_to_observed.png`
- `./figs/limit_dep_tobit_fig02_latent_observed_distribution.png`
- `./figs/limit_dep_tobit_fig03_x_to_latent_demand.png`
- `./figs/limit_dep_tobit_fig04_marginal_effects.png`

In [1]:
# ============================================================
# 0. 全局设置
# ============================================================

import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats
from scipy.optimize import minimize
import statsmodels.api as sm

FIG_DIR = Path("./figs")
DATA_DIR = Path("./data")
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# 中文字体设置：Windows 本地可优先使用 SimHei 或 Microsoft YaHei；
# Linux/Quarto 环境中优先使用 Noto Sans CJK。
font_paths = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc",
    "C:/Windows/Fonts/simhei.ttf",
    "C:/Windows/Fonts/msyh.ttc",
]

for fp in font_paths:
    if os.path.exists(fp):
        fm.fontManager.addfont(fp)
        font_prop = fm.FontProperties(fname=fp)
        plt.rcParams["font.family"] = font_prop.get_name()
        break

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

def savefig(fig, name):
    """
    同时保存 png 和 svg，便于 Quarto、网页和推文使用。
    """
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    plt.close(fig)

rng = np.random.default_rng(20260429)

In [2]:
# ============================================================
# 1. 生成企业贷款金额的 Tobit 模拟数据
# ============================================================

def simulate_tobit_credit(n=3000, seed=20260429):
    """
    生成 Tobit 章节使用的企业信贷数据。

    B_star 是潜在净借款需求，不是实际贷款金额。
    当 B_star <= 0 时，实际银行贷款金额被记录为 0。
    """
    r = np.random.default_rng(seed)

    # 投资机会：可理解为销售增长、Tobin's Q 或 ROA 与贷款成本之差的综合指标
    opportunity = r.normal(0, 1, n)

    # 抵押能力：先生成 0-1 变量，再标准化
    collateral_raw = r.beta(2.2, 2.0, n)
    collateral = (collateral_raw - collateral_raw.mean()) / collateral_raw.std()

    # 内部现金：现金越充裕，对银行贷款的净需求越弱
    cash_raw = r.beta(2.0, 4.5, n)
    cash = (cash_raw - cash_raw.mean()) / cash_raw.std()

    # 扩展变量：风险和规模，用于让数据更接近真实情形
    risk = r.normal(0, 1, n)
    size = r.normal(0, 1, n)

    # 潜在净借款需求
    u = r.normal(0, 0.85, n)
    B_star = (
        -0.30
        + 0.95 * opportunity
        + 1.10 * collateral
        - 0.70 * cash
        + 0.25 * size
        - 0.25 * risk
        + u
    )

    # 观测贷款金额：不能小于 0
    loan_amt = np.maximum(0, B_star) * 80.0
    loan_pos = (loan_amt > 0).astype(int)

    df = pd.DataFrame({
        "firm_id": np.arange(1, n + 1),
        "opportunity": opportunity,
        "collateral": collateral,
        "cash": cash,
        "risk": risk,
        "size": size,
        "B_star": B_star,
        "loan_amt": loan_amt,
        "loan_pos": loan_pos,
    })
    return df

df = simulate_tobit_credit()
df.to_csv(DATA_DIR / "tobit_credit_sim.csv", index=False, encoding="utf-8-sig")

df.head()

,firm_id,opportunity,collateral,cash,risk,size,B_star,loan_amt,loan_pos
0,1,-0.565891,1.312192,1.612859,0.813688,0.066029,-1.869075,0.000000,0
1,2,0.402974,1.509527,-0.062144,-0.937793,0.495055,2.617716,209.417276,1
2,3,-0.978873,0.437743,3.341655,0.110751,1.605443,-3.082948,0.000000,0
3,4,0.571814,-0.386959,0.429776,-0.208431,-0.123657,0.340512,27.240925,1
4,5,-1.339470,-0.524515,-1.213267,-1.892202,-1.350724,-1.237195,0.000000,0


In [3]:
# ============================================================
# 2. 图 1：潜在净借款需求与观测贷款金额
# ============================================================

x = np.linspace(-3, 4, 300)
y = np.maximum(0, x)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.plot(x, y, lw=2.5, label=r"$B=\max(0,B^*)$")
ax.axvline(0, ls="--", lw=1.2, color="0.3")
ax.axhline(0, lw=1.0, color="0.3")

ax.text(-2.8, 0.35, "潜在净借款需求为负\n观测贷款金额归并为 0", fontsize=11)
ax.text(1.1, 2.7, "潜在净借款需求为正\n观测到正贷款金额", fontsize=11)

ax.set_xlabel(r"潜在净借款需求 $B^*$")
ax.set_ylabel(r"观测银行贷款金额 $B$")
ax.set_title("Tobit 机制：潜在净借款需求被归并到 0")
ax.legend(frameon=False)

savefig(fig, "limit_dep_tobit_fig01_latent_to_observed")

In [4]:
# ============================================================
# 3. 图 2：潜在变量与观测变量的分布
# ============================================================

fig, ax = plt.subplots(figsize=(7.4, 4.3))

ax.hist(
    df["B_star"],
    bins=45,
    alpha=0.50,
    density=True,
    label=r"潜在净借款需求 $B^*$"
)

ax.hist(
    df["loan_amt"] / 80.0,
    bins=45,
    alpha=0.45,
    density=True,
    label=r"观测贷款金额 $B/80$"
)

ax.axvline(0, ls="--", color="0.3", lw=1.1)
ax.set_xlabel("标准化金额")
ax.set_ylabel("密度")
ax.set_title("归并后，0 点附近出现大量堆积")
ax.legend(frameon=False)

savefig(fig, "limit_dep_tobit_fig02_latent_observed_distribution")

In [5]:
# ============================================================
# 4. 图 3：x 如何影响潜在净借款需求
# ============================================================

grid = np.linspace(-2.5, 2.5, 120)

fig, ax = plt.subplots(figsize=(7.4, 4.3))

for cval, label in [
    (-1.0, "抵押能力较低"),
    (0.0,  "抵押能力中等"),
    (1.0,  "抵押能力较高"),
]:
    xb = -0.30 + 0.95 * grid + 1.10 * cval
    ax.plot(grid, xb, lw=2.1, label=label)

ax.axhline(0, ls="--", color="0.3", lw=1.1)
ax.set_xlabel("投资机会 opportunity")
ax.set_ylabel(r"潜在净借款需求 $E(B^*|x)$")
ax.set_title("同一组 x 同时影响是否为 0 与正值大小")
ax.legend(frameon=False)

savefig(fig, "limit_dep_tobit_fig03_x_to_latent_demand")

In [6]:
# ============================================================
# 5. 轻量 Tobit MLE 函数
# ============================================================

def fit_tobit_left0(y, X):
    """
    估计左归并点为 0 的 Tobit 模型。

    参数向量最后一项为 log(sigma)，这样可以保证 sigma > 0。
    """
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)

    def nll(params):
        beta = params[:-1]
        sigma = np.exp(params[-1])
        xb = X @ beta

        # 左归并点为 0
        z0 = (0 - xb) / sigma
        is_zero = y <= 1e-12

        ll = np.empty_like(y)
        ll[is_zero] = stats.norm.logcdf(z0[is_zero])
        ll[~is_zero] = (
            stats.norm.logpdf((y[~is_zero] - xb[~is_zero]) / sigma)
            - np.log(sigma)
        )
        return -np.sum(ll)

    # 用 OLS 结果作为初值，提高数值稳定性
    ols_res = sm.OLS(y, X).fit()
    start = np.r_[ols_res.params, np.log(np.std(ols_res.resid) + 1e-6)]

    res = minimize(
        nll,
        start,
        method="BFGS",
        options={"maxiter": 2000}
    )
    return res

# 使用未乘以 80 的 loan_scaled 估计，便于数值稳定
y_scaled = df["loan_amt"].to_numpy() / 80.0
X = sm.add_constant(df[["opportunity", "collateral", "cash"]])

tobit_res = fit_tobit_left0(y_scaled, X)
beta_hat = tobit_res.x[:-1]
sigma_hat = np.exp(tobit_res.x[-1])

est = pd.DataFrame({
    "term": X.columns,
    "estimate": beta_hat
})
est["sigma"] = sigma_hat
est.to_csv(DATA_DIR / "tobit_credit_estimates.csv", index=False, encoding="utf-8-sig")

est

,term,estimate,sigma
0,const,-0.240997,0.886069
1,opportunity,0.954613,0.886069
2,collateral,1.084940,0.886069
3,cash,-0.640374,0.886069


In [7]:
# ============================================================
# 6. 图 4：三类边际效应
# ============================================================

grid = np.linspace(-2.5, 2.5, 150)
collateral0 = 0.0
cash0 = 0.0

Xg = np.column_stack([
    np.ones_like(grid),
    grid,
    np.full_like(grid, collateral0),
    np.full_like(grid, cash0)
])

xb = Xg @ beta_hat
z = xb / sigma_hat

phi = stats.norm.pdf(z)
Phi = stats.norm.cdf(z)
lam = phi / np.maximum(Phi, 1e-10)

j = list(X.columns).index("opportunity")
b_j = beta_hat[j]

# 对正值概率的边际效应
me_prob = phi * b_j / sigma_hat

# 对非条件期望 E(B|x) 的边际效应
me_uncond = Phi * b_j

# 对正值条件均值 E(B|B>0,x) 的边际效应
me_cond = b_j * (1 - lam * (z + lam))

fig, ax = plt.subplots(figsize=(7.4, 4.3))

ax.plot(grid, me_uncond, lw=2.2, label="对非条件期望的边际效应")
ax.plot(grid, me_prob, lw=2.0, ls="--", label="对正贷款概率的边际效应")
ax.plot(grid, me_cond, lw=2.0, ls=":", label="对正值条件均值的边际效应")

ax.set_xlabel("投资机会 opportunity")
ax.set_ylabel("边际效应")
ax.set_title("Tobit 的边际效应不是只有一种")
ax.legend(frameon=False)

savefig(fig, "limit_dep_tobit_fig04_marginal_effects")

# 运行完成检查

如果上述代码顺利运行，当前目录下应该出现：

- `./data/tobit_credit_sim.csv`
- `./data/tobit_credit_estimates.csv`
- `./figs/limit_dep_tobit_fig01_latent_to_observed.png`
- `./figs/limit_dep_tobit_fig02_latent_observed_distribution.png`
- `./figs/limit_dep_tobit_fig03_x_to_latent_demand.png`
- `./figs/limit_dep_tobit_fig04_marginal_effects.png`